## Imports

Imports

In [1]:
import tvm
import tvm.testing
from tvm.relay import testing
from tvm import relax, relay
from tvm.relax.testing import relay_translator, nn
from tvm.runtime import vm as vm_rt
from tvm.script import relax as R
import numpy as np

[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled with support for Arm(R)-based targets.
[15:00:18] /var/tmp/ga87puy/tvm_relax/src/target/parsers/aprofile.cc:97: Warning: Cannot parse target features. LLVM was not compiled w

## Config

In [2]:
# Target
# TARGET = "llvm"
TARGET = "c"
# TARGET = tvm.target.Target("c", host="c")
target = tvm.target.Target(TARGET, host=TARGET)

# Pipeline (Relax only)
# PIPELINE = "default"
PIPELINE = "micro"
default_pipeline = "default_build"
micro_pipeline = "micro2_build"

# Exec mode (Relax only)
# EXEC_MODE = "bytecode"
# EXEC_MODE = "compiled"
EXEC_MODE = "crt"
bytecode_exec_mode = "bytecode"
compiled_exec_mode = "compiled"
crt_exec_mode = "crt"

# Executor (Relay only)
UNPACKED = False
# UNPACKED = True
USMP = False
# EXECUTOR = "graph"
EXECUTOR = "aot"
aot_executor = tvm.relay.backend.Executor("aot", {"interface-api": "c" if UNPACKED else "packed", "unpacked-api": UNPACKED})
graph_executor = tvm.relay.backend.Executor("graph", {"link-params": False})

# Runtime (Relay only?)
# RUNTIME = "cpp"
RUNTIME = "crt"
SYSTEM_LIB = not UNPACKED
cpp_runtime = tvm.relay.backend.Runtime("cpp", {"system-lib": SYSTEM_LIB})
crt_runtime = tvm.relay.backend.Runtime("crt", {"system-lib": SYSTEM_LIB})

# Pass Config (Relay only)
USMP = False
FUSE_DEPTH = 1
VECTORIZE = False
pass_config = {"tir.usmp.enable": USMP, "relay.FuseOps.max_depth": FUSE_DEPTH, "tir.disable_vectorize": not VECTORIZE}

## Define Models

### Common

Matmul

In [3]:
matmul_input_size = 64
matmul_hidden_size = 10
matmul_output_size = 4
    
matmul_dtype = "float32"
    
matmul_weights_matrix = np.random.random((matmul_hidden_size, matmul_output_size)).astype(matmul_dtype)
matmul_bias_matrix = np.random.random((matmul_output_size,)).astype(matmul_dtype)

matmul_params = {"weights": tvm.nd.array(matmul_weights_matrix), "bias": tvm.nd.array(matmul_bias_matrix)}
matmul_data = tvm.nd.array(np.random.rand(matmul_input_size, matmul_hidden_size).astype(matmul_dtype))

Conv2d

In [4]:
conv2d_input_n = 1
conv2d_input_c = 16
conv2d_input_h = 64
conv2d_input_w = 64
conv2d_kernel_h = 4
conv2d_kernel_w = 4
conv2d_kernel_ci = 16
conv2d_kernel_co = 16
conv2d_output_n = 1
conv2d_output_c = 16
conv2d_output_h = 61
conv2d_output_w = 61
    
conv2d_dtype = "float32"
    
conv2d_weights_matrix = np.random.random((conv2d_kernel_h, conv2d_kernel_w, conv2d_kernel_ci, conv2d_kernel_co)).astype(conv2d_dtype)
conv2d_bias_matrix = np.random.random((conv2d_output_w,)).astype(conv2d_dtype)

### Relax

Define Matmul model (Relax)

In [5]:
def relax_dense():
    builder = relax.BlockBuilder()
    
    with builder.function("main"):
        input = relax.Var("x", R.Tensor((matmul_input_size, matmul_hidden_size), matmul_dtype))
        weights = relax.Constant(tvm.nd.array(matmul_weights_matrix))
        bias = relax.Constant(tvm.nd.array(matmul_bias_matrix))
        output_matmul = relax.op.matmul(input, weights)
        output_bias = relax.op.add(output_matmul, bias)
        builder.emit_func_output(output_bias, params=[input])
    return builder.get(), matmul_data, matmul_params

Define Conv2d model (Relax)

In [6]:
def relax_conv2d():
    builder = relax.BlockBuilder()
    
    with builder.function("main"):
        input = relax.Var("x", R.Tensor((input_n, input_c, input_h, input_w), conv2d_dtype))
        weights = relax.Constant(tvm.nd.array(weights_matrix))
        bias = relax.Constant(tvm.nd.array(bias_matrix))
        output_conv2d = relax.op.nn.conv2d(input, weights, data_layout="NCHW", kernel_layout="HWIO")
        output_bias = relax.op.add(output_conv2d, bias)
        builder.emit_func_output(output_bias, params=[input])
    
    return builder.get(), conv2d_data, conv2d_params

### Relay

Define Matmul model (Relay)

In [7]:
def relay_dense():

    x = relay.var("x", shape=(matmul_input_size, matmul_hidden_size), dtype=matmul_dtype)
    weight = relay.const(tvm.nd.array(matmul_weights_matrix))
    bias = relay.const(tvm.nd.array(matmul_bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func), matmul_data, matmul_params

Define Conv2D model (Relay)

In [8]:
_ = """
def relay_conv2d():
    dtype = "float32"  #  TODO: int32

    weights_matrix = np.random.random((hidden_size, output_size)).astype(dtype)
    bias_matrix = np.random.random((output_size,)).astype(dtype)

    x = relay.var("x", shape=(input_size, hidden_size), dtype=dtype)
    weight = relay.const(tvm.nd.array(weights_matrix))
    bias = relay.const(tvm.nd.array(bias_matrix))
    output_matmul = relay.nn.matmul(x, weight)
    output_bias = relay.op.add(output_matmul, bias)
    func = relay.Function(relay.analysis.free_vars(output_bias), output_bias)
    return tvm.IRModule.from_expr(func)
    return func, conv2d_data, conv2d_params
"""

## Show models

Pick used models

In [9]:
# -- Relax --
relax_mod, relax_data, relax_params = relax_dense()
# relax_mod = relax_conv2d()

# -- Relay --
relay_mod, relay_data, relay_params = relay_dense()
# relay_mod = relay_conv2d()

Show Relax module

In [10]:
relax_mod.show()

Show Relay module

In [11]:
relay_mod.show()

## Instruments

Define Pass Instrument to look at intermediate IRs during build

In [12]:
@tvm.instrument.pass_instrument
class MyInstrument:

    def __init__(self):
        self.skip_pass_name = []
        self.output = []
        self.output_after = []
        self.idx = 0

    def run_before_pass(self, mod, pass_info):
        self.idx += 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        # print(self.idx * "  " + ">", self.idx, pass_info.name, g_, len(self.output))
        # print(dir(mod))
        
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        # tmp = (mod.script(show_meta=True), str(pass_info))
        tmp = (mod, pass_info)
        self.output.append(tmp)


    def run_after_pass(self, mod, pass_info):
        self.idx -= 1
        handle = mod.handle
        g = dict(mod.global_var_map_)
        g_ = list(g.keys())
        if len(g_) == 0:
            return
        # print((self.idx + 1) * "  " + "<", self.idx + 1, pass_info.name, g_, len(self.output_after))
        # print(dir(mod))
        # tmp = (mod.astext(show_meta_data=True), str(pass_info))
        # tmp = (str(mod), str(pass_info))
        tmp = (mod, pass_info)
        self.output_after.append(tmp)
        pass

## Build

### Relax

Relax build (VM Bytecode LLVM)

In [13]:
if PIPELINE == "default" and EXEC_MODE == "bytecode" and TARGET == "llvm":
    relax_instrument_ex_vm_bytecode_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_ex_vm_bytecode_llvm]):
        ex_vm_bytecode_llvm = relax.build(relax_mod, target=target, pipeline=default_pipeline, exec_mode=bytecode_exec_mode)
else:
    relax_instrument_ex_vm_bytecode_llvm = None
    ex_vm_bytecode_llvm = None

Relax build (VM Compiled LLVM)

In [14]:
if PIPELINE == "default" and EXEC_MODE == "compiled" and TARGET == "llvm":
    relax_instrument_ex_vm_compiled_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_ex_vm_compiled_llvm]):
        ex_vm_compiled_llvm = relax.build(relax_mod, target=target, pipeline=default_pipeline, exec_mode=compiled_exec_mode)
else:
    relax_instrument_ex_vm_compiled_llvm = None
    ex_vm_compiled_llvm = None

Relax build (AOT C++ LLVM)

In [15]:
if PIPELINE == "micro" and EXEC_MODE == "crt" and TARGET == "llvm" and RUNTIME == "cpp":
    relax_instrument_ex_aot_cpp_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_ex_aot_cpp_llvm]):
        ex_aot_cpp_llvm = relax.build(relax_mod, target=target, pipeline=micro_pipeline, exec_mode=crt_exec_mode, executor=aot_executor, runtime=cpp_runtime)
else:
    relax_instrument_ex_aot_cpp_llvm = None
    ex_aot_cpp_llvm = None

Relax build (AOT CRT LLVM)

In [16]:
if PIPELINE == "micro" and EXEC_MODE == "crt" and TARGET == "llvm" and RUNTIME == "crt":
    relax_instrument_ex_aot_crt_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_ex_aot_crt_llvm]):
        ex_aot_crt_llvm = relax.build(relax_mod, target=target, pipeline=micro_pipeline, exec_mode=crt_exec_mode, executor=aot_executor, runtime=crt_runtime)
else:
    relax_instrument_ex_aot_crt_llvm = None
    ex_aot_crt_llvm = None

Relax build (AOT CRT C)

In [17]:
if PIPELINE == "micro" and EXEC_MODE == "crt" and TARGET == "c" and RUNTIME == "crt":
    relax_instrument_ex_aot_crt_c = MyInstrument()
    with tvm.transform.PassContext(instruments=[relax_instrument_ex_aot_crt_c]):
        ex_aot_crt_c = relax.build(relax_mod, target=target, pipeline=micro_pipeline, exec_mode=crt_exec_mode, executor=aot_executor, runtime=crt_runtime)
else:
    relax_instrument_ex_aot_crt_c = None
    ex_aot_crt_c = None

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1194: EmitNormalCall

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1203: defined && kPackedFunc

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1207: defined && kPackedFunc && prim_func

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1194: EmitNormalCall

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1203: defined && kPackedFunc

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_crt_tir.cc:1207: defined && kPackedFunc && prim_func

[15:00:19] /var/tmp/ga87puy/tvm_relax/src/driver/driver_api.cc:618: unpacked_api=0


unpacked_api 0 <class 'int'>
executor aot{"interface-api": "packed", "link-params": T.bool(True), "unpacked-api": T.bool(False)}


HERE! 


metadata012 MetadataObj(0x388fb70)
func_metadata['__tvm_main__'] FunctionInfoNode(
workspace_sizes={c -keys=cpu : 1024},
  io_sizes={c -keys=cpu : 3584},
  constant_sizes={c -keys=cpu : 352},
  tir_primfuncs={c -keys=cpu : # from tvm.script import tir as T

@T.prim_func
def tvmgen_default___tvm_main__(x: T.handle, output: T.handle):
    T.func_attr({"input_vars": [x], "output_vars": [output], "runner_function": T.bool(True), "target": T.target({"keys": ["cpu"], "kind": "c", "tag": ""}), "tir.is_entry_func": T.bool(True)})
    x_buffer = T.match_buffer(x, (T.int64(64), T.int64(10)), align=16)
    output_buffer = T.match_buffer(output, (T.int64(64), T.int64(4)), align=16)
    constant_0 = T.allocate_const([0.85998880863189697, 0.88502389192581177, 0.82317811250686646, 0.64045882225036621, 0.066371634602546692, 0.79834097623825073, 0.061610225588083267, 0.64703601598739624, 0.31827205419540405, 0.50206494331359863, 0.67880856990814209, 0.51086336374282837, 0.11074915528297424, 0.908376812

[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:1107: runtime.CreateCSourceCrtMetadataModule
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:981: CreateCSourceCrtMetadataModule
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:674: CreateSource
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:679: metadata_.defined()=0
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relax/backend/vm/codegen_vm.cc:487: metadata123=MetadataObj(0x388fb70)
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/metadata_module.cc:206: CreateMetadataModule
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/metadata_module.cc:207: runtime->name=crt
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/metadata_module.cc:48: CreateCrtMetadataModule
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:981: CreateCSourceCrtMetadataModule
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/target/source/source_module.cc:984: m

### Relay

Relay build (Graph C++ LLVM)

In [18]:
if RUNTIME == "cpp" and EXECUTOR == "graph" and TARGET == "llvm":
    relay_instrument_lib_graph_cpp_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_lib_graph_cpp_llvm], config=pass_config):
        lib_graph_cpp_llvm = relay.build(relay_mod, target=target, runtime=cpp_runtime, executor=graph_executor)
else:
    relay_instrument_lib_graph_cpp_llvm = None
    lib_graph_cpp_llvm = None

Relay build (AoT C++ LLVM)

In [19]:
if RUNTIME == "cpp" and EXECUTOR == "aot" and TARGET == "llvm" and not UNPACKED and not USMP:
    relay_instrument_lib_aot_cpp_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_lib_aot_cpp_llvm], config=pass_config):
        lib_aot_cpp_llvm = relay.build(relay_mod, target=target, runtime=cpp_runtime, executor=aot_executor)
else:
    relay_instrument_lib_aot_cpp_llvm = None
    lib_aot_cpp_llvm = None

Relay build (Graph CRT, Packed API, LLVM)

In [20]:
if RUNTIME == "crt" and EXECUTOR == "graph" and TARGET == "llvm" and not UNPACKED and not USMP:
    relay_instrument_crt_lib_graph = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_crt_lib_graph], config=pass_config):
        crt_lib_graph = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=graph_executor)
else:
    relay_instrument_crt_lib_graph = None
    crt_lib_graph = None

Relay build (Graph CRT, Packed API, C)

In [21]:
if RUNTIME == "crt" and EXECUTOR == "graph" and TARGET == "c" and not UNPACKED and not USMP:
    relay_instrument_crt_lib_graph = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_crt_lib_graph], config=pass_config):
        crt_lib_graph = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=graph_executor)
else:
    relay_instrument_crt_lib_graph = None
    crt_lib_graph = None

Relay build (AOT CRT, Packed API, LLVM)

In [22]:
if RUNTIME == "crt" and EXECUTOR == "aot" and TARGET == "llvm" and not UNPACKED and not USMP:
    relay_instrument_lib_aot_crt_llvm = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_lib_aot_crt_llvm], config=pass_config):
        lib_aot_crt_llvm = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=aot_executor)
else:
    relay_instrument_lib_aot_crt_llvm = None
    lib_aot_crt_llvm = None

Relay build (AOT CRT, Packed API, C)

In [23]:
if RUNTIME == "crt" and EXECUTOR == "aot" and TARGET == "c" and not UNPACKED and not USMP:
    relay_instrument_lib_aot_crt_c = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_lib_aot_crt_c], config=pass_config):
        lib_aot_crt_c = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=aot_executor)
else:
    relay_instrument_lib_aot_crt_c = None
    lib_aot_crt_c = None

[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:311: Build
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:429: BuildRelay
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:430: relay_module=def @main(%x: Tensor[(64, 10), float32]) {
  %0 = nn.matmul(%x, meta[relay.Constant][0], units=None);
  add(%0, meta[relay.Constant][1])
}


[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:433: relay_module=def @main(%x: Tensor[(64, 10), float32]) {
  %0 = nn.matmul(%x, meta[relay.Constant][0], units=None);
  add(%0, meta[relay.Constant][1])
}


[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:340: OptimizeImpl
[15:00:21] /var/tmp/ga87puy/tvm_relax/src/relay/backend/build_module.cc:435: relay_module=def @main(%x {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=3a66310, kind='c', keys={'cpu'}, host=Target(id=3a6e830, kind='c', keys={'cpu'})))}: Tensor

Relay build (AOT CRT, Unpacked API, C)

In [24]:
if RUNTIME == "crt" and EXECUTOR == "aot" and TARGET == "c" and UNPACKED and not USMP:
    relay_instrument_lib_aot_crt_c_unpacked = MyInstrument()
    with tvm.transform.PassContext(instruments=[relay_instrument_lib_aot_crt_c_unpacked], config=pass_config):
        lib_aot_crt_c_unpacked = relay.build(relay_mod, target=target, runtime=crt_runtime, executor=aot_executor)
else:
    relay_instrument_lib_aot_crt_c_unpacked = None
    lib_aot_crt_c_unpacked = None

Relay build (AOT CRT, Unpacked API, C, USMP)

In [25]:
if RUNTIME == "crt" and EXECUTOR == "aot" and TARGET == "c" and UNPACKED and USMP:
    raise NotImplementedError

Pick compiled module

In [26]:
# -- Relax --
if RUNTIME == "cpp":
    if TARGET == "llvm":
        if PIPELINE == "default" and EXEC_MODE == "bytecode":
            ex, relax_instrument = ex_vm_bytecode_llvm, relax_instrument_ex_vm_bytecode_llvm
        elif PIPELINE == "default" and EXEC_MODE == "compiled":
            ex, relax_instrument = ex_vm_compiled_llvm, relax_instrument_ex_vm_compiled_llvm
        elif PIPELINE == "micro" and EXEC_MODE == "crt":
            ex, relax_instrument = ex_aot_cpp_llvm, relax_instrument_ex_aot_cpp_llvm
        else:
            assert False, f"Invalid PIPELINE ({PIPELINE}) and EXEC_MODE ({EXEC_MODE}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    else:
        assert False, f"Invalid TARGET ({TARGET}) for RUNTIME ({RUNTIME})"
elif RUNTIME == "crt":
    if TARGET == "llvm":
        if PIPELINE == "micro" and EXEC_MODE == "crt":
            ex, relax_instrument = ex_aot_crt_llvm, relax_instrument_ex_aot_crt_llvm
        else:
            assert False, f"Invalid PIPELINE ({PIPELINE}) and EXEC_MODE ({EXEC_MODE}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    elif TARGET == "c":
        if PIPELINE == "micro" and EXEC_MODE == "crt":
            ex, relax_instrument = ex_aot_crt_c, relax_instrument_ex_aot_crt_c
        else:
            assert False, f"Invalid PIPELINE ({PIPELINE}) and EXEC_MODE ({EXEC_MODE}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    else:
        assert False, f"Invalid TARGET ({TARGET}) for RUNTIME ({RUNTIME})"
else:
    assert False, f"Invalid RUNTIME ({RUNTIME})"

# -- Relay --
if RUNTIME == "cpp":
    if TARGET == "llvm":
        if EXECUTOR == "graph" and not UNPACKED and not USMP:
            lib, relay_instrument = lib_graph_cpp_llvm, relay_instrument_lib_graph_cpp_llvm
        elif EXECUTOR == "aot" and not UNPACKED and not USMP:
            lib, relay_instrument = lib_aot_cpp_llvm, relay_instrument_lib_aot_cpp_llvm
        else:
            assert False, f"Invalid EXECUTOR ({EXECUTOR}), UNPACKED ({UNPACKED}) and USMP ({USMP}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    else:
        assert False, f"Invalid TARGET ({TARGET}) for RUNTIME ({RUNTIME})"
elif RUNTIME == "crt":
    if TARGET == "llvm":
        if EXECUTOR == "graph" and not UNPACKED and not USMP:
            lib, relay_instrument = ib_graph_crt_llvm, relay_instrument_lib_graph_crt_llvm
        elif EXECUTOR == "aot" and not UNPACKED and not USMP:
            lib, relay_instrument = lib_aot_crt_llvm, relay_instrument_lib_aot_crt_llvm
        else:
            assert False, f"Invalid EXECUTOR ({EXECUTOR}), UNPACKED ({UNPACKED}) and USMP ({USMP}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    elif TARGET == "c":
        if EXECUTOR == "graph" and not UNPACKED and not USMP:
            lib, relay_instrument = lib_graph_crt_c, relay_instrument_lib_graph_crt_c
        elif EXECUTOR == "aot" and not UNPACKED and not USMP:
            lib, relay_instrument = lib_aot_crt_c, relay_instrument_lib_aot_crt_c
        elif EXECUTOR == "aot" and UNPACKED and not USMP:
            lib, relay_instrument = lib_aot_crt_c_unpacked, relay_instrument_lib_aot_crt_c_unpacked
        elif EXECUTOR == "aot" and UNPACKED and USMP:
            lib, relay_instrument = lib_aot_crt_c_unpacked_usmp, relay_instrument_lib_aot_crt_c_unpacked_usmp
        else:
            assert False, f"Invalid EXECUTOR ({EXECUTOR}), UNPACKED ({UNPACKED}) and USMP ({USMP}) for RUNTIME ({RUNTIME}) and TARGET ({TARGET})"
    else:
        assert False, f"Invalid TARGET ({TARGET}) for RUNTIME ({RUNTIME})"
else:
    assert False, f"Invalid RUNTIME ({RUNTIME})"

## Investigate

Look at generated C code (Relax)

In [27]:
if RUNTIME == "crt" and TARGET == "c":
    dso_mods = ex.lib._collect_dso_modules()
    for dso in dso_mods:
        print(dso.get_source())

#include "tvm/runtime/c_runtime_api.h"
#ifdef __cplusplus
extern "C" {
#endif
TVM_DLL int32_t tvmgen_default___tvm_main__(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);

int32_t tvmgen_default_run(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle) {
TVMValue tensors[2];
tensors[0] = ((TVMValue*)args)[0];
tensors[1] = ((TVMValue*)args)[1];
return tvmgen_default___tvm_main__((void*)tensors, type_code, num_args, out_value, out_type_code, resource_handle);
}
#ifdef __cplusplus
}
#endif
;
#include <inttypes.h>
#include <tvm/runtime/metadata_types.h>
#include <tvm/runtime/c_runtime_api.h>
static const int64_t kTvmgenMetadata_inputs_0_shape[2] = {
64L, 
10L};
static const struct TVMTensorInfo kTvmgenMetadata_inputs[1] = {
{
"x" /* name*/, 
kTvmgenMetadata_inputs_0_shape, 
2L /* num_shape*/, 
{2, 32, 1} /* dtype*/}
};
static const int64_t kTvmgenMetadata_outputs_0_shape[2]

Look at generated C code (Relay)

In [28]:
if RUNTIME == "crt" and TARGET == "c":
    dso_mods = lib.lib._collect_dso_modules()
    for dso in dso_mods:
        print(dso.get_source())

#include <tvm/runtime/crt/module.h>
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_run(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_fused_add(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_fused_nn_matmul(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default___tvm_main__(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);
#ifdef __cplusplus
extern "C"
#endif
TVM_DLL int32_t tvmgen_default_get_c_metadata(TVMValue* args, int* type_code, int num_args, TVMValue* out_value, int* out_type_code, void* resource_handle);
static TVMB

Generate MLF (Relax)

In [29]:
from tvm.micro import export_model_library_format
mlf_dest = "/tmp/relax_mlf.tar"
export_model_library_format([ex], mlf_dest)

mod.target c -keys=cpu 
main_relax_func # from tvm.script import relax as R

@R.function
def main(x: R.Tensor((64, 10), dtype="float32")) -> R.Tensor((64, 4), dtype="float32"):
    R.func_attr({"relax.force_pure": 1})
    alloc: R.Tensor((64, 4), dtype="float32") = R.builtin.alloc_tensor(R.shape([64, 4]), R.dtype("float32"), R.prim_value(0), R.str("global"))
    matmul(x, metadata["relax.expr.Constant"][0], alloc)
    gv: R.Tensor((64, 4), dtype="float32") = alloc
    alloc1: R.Tensor((64, 4), dtype="float32") = R.builtin.alloc_tensor(R.shape([64, 4]), R.dtype("float32"), R.prim_value(0), R.str("global"))
    add(gv, metadata["relax.expr.Constant"][1], alloc1)
    gv1: R.Tensor((64, 4), dtype="float32") = alloc1
    return gv1

# Metadata omitted. Use show_meta=True in script() method to show it.
dir(main_relax_func) ['__annotations__', '__call__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__

?! 


main_relax_func.params[0] x <class 'tvm.relax.expr.Var'> ['__add__', '__annotations__', '__call__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__pow__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmod__', '__rmul__', '__rpow__', '__rsub__', '__rtruediv__', '__setattr__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__sub__', '__subclasshook__', '__truediv__', '__weakref__', '_check_for_tensor_struct_info', '_checked_type_', '_move', '_relax_script', 'astype', 'byte_offset', 'checked_type', 'dtype', 'elem_offset', 'handle', 'legacy_repr', 'name_hint', 'ndim', 'same_as', 'script', 'shape', 'show', 'span', 'strides', 'struct_info

PosixPath('/tmp/relax_mlf.tar')

Generate MLF (Relay)

In [30]:
from tvm.micro import export_model_library_format
mlf_dest = "/tmp/relay_mlf.tar"
export_model_library_format([lib], mlf_dest)

mod.target [c -keys=cpu ]
main_relay_func.params[0] free_var %x {virtual_device=VirtualDevice(device_type=1, virtual_device_id=0, target=Target(id=31cff10, kind='c', keys={'cpu'}, host=Target(id=31d49c0, kind='c', keys={'cpu'})))}: Tensor[(64, 10), float32] /* ty=Tensor[(64, 10), float32] */;
%x <class 'tvm.relay.expr.Var'> ['__add__', '__annotations__', '__call__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__div__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_handle_by_constructor__', '__init_subclass__', '__le__', '__lt__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__radd__', '__rdiv__', '__reduce__', '__reduce_ex__', '__repr__', '__rmul__', '__rsub__', '__rtruediv__', '__setattr__', '__setstate__', '__sizeof__', '__slots__', '__str__', '__sub__', '__subclasshook__', '__truediv__', '__weakref__', '_checked_type_', '_move', 'astext', 'astype', 'checked_typ

PosixPath('/tmp/relay_mlf.tar')

Intermediate IRs (Relax)

In [ ]:
relax_instrument.output[-1][0].show()

Intermediate IRs (Relay)

In [46]:
relay_instrument.output[-1][0].show()

## Run

### Relax

In [153]:
if TARGET == "llvm":
    dev = tvm.device(str(TARGET), dev_id=0)
    if EXEC_MODE in ["bytecode", "compiled"]:
        # vm = relax.VirtualMachine(ex, tvm.cpu())
        vm = relax.VirtualMachine(ex, dev)
        relax_output = vm["main"](relax_data).numpy()
    elif EXEC_MODE == "crt":
        factory_module = ex
        rt_mod = tvm.runtime.executor.AotModule(factory_module["default"](dev))
        rt_mod.set_input("x", relax_data)
        rt_mod.run()
        relax_output = rt_mod.get_output(0).numpy()
    else:
        assert False
else:
    relax_output = None
    print("C target does not support execution")

C target does not support execution


Show Result

In [52]:
# relax_output

### Relay

In [53]:
if TARGET == "llvm":
    dev = tvm.device(str(TARGET), dev_id=0)
    if EXECUTOR == "graph":
        rt_mod = tvm.contrib.graph_executor.GraphModule(lib["default"](dev))
    elif EXECUTOR == "aot":
        rt_mod = tvm.runtime.executor.AotModule(lib["default"](dev))
    else:
        assert False
    rt_mod.set_input("x", relay_data)
    rt_mod.run()
    relay_output = rt_mod.get_output(0).numpy()
else:
    relay_output = None
    print("C target does not support execution")

[10:24:17] /var/tmp/ga87puy/tvm_relax/src/runtime/system_library.cc:41: Warning: SystemLib symbol __tvm_module_ctx get overriden to a different address 0x7f24942d9000->0x7f24942dc000
[10:24:17] /var/tmp/ga87puy/tvm_relax/src/runtime/system_library.cc:41: Warning: SystemLib symbol __tvm_module_ctx get overriden to a different address 0x7f24942d5000->0x7f24942d9000
[10:24:17] /var/tmp/ga87puy/tvm_relax/src/runtime/system_library.cc:41: Warning: SystemLib symbol tvmgen_default___tvm_main__ get overriden to a different address 0x7f24942d6000->0x7f24942ddd20


Show Result

In [54]:
# relay_output

## Compare

In [55]:
assert not(relax_output is None or relay_output is None)
tvm.testing.assert_allclose(relax_output, relay_output, rtol=1e-4, atol=1e-4)